In [ ]:
# Reset env to erase previous vars, dicts, dfs, etc.
%reset -f

from datetime import datetime

print(f"Environment reset | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# Configuration & imports
import os
import pandas as pd
import numpy as np
from collections import defaultdict
from datetime import datetime
import re
from collections import OrderedDict
import chardet
import openpyxl

print(f"Configuration & imports done | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# Helper to remove previous progress message
def print_overwrite(msg, width=120):
    print(" " * width, end="\r")   # clear line
    print(msg, end="\r", flush=True)

print(f"Helper to remove previous progress message created | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# Specify CSV path
# Change this to point to the CSV file
CSV_PATH = r"C:\Users\Count_Dracula\FooDB\FooDB.csv"

# Quick existence check
assert os.path.exists(CSV_PATH), f"Content CSV file not found: {CSV_PATH}"
print(f"Content CSV path is valid | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# Reading CSV file
print(f"Reading Content CSV with pandas | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"It will take a while...\n")

# Read entire CSV
content_df = pd.read_csv(CSV_PATH, low_memory=False)

print(f"Content CSV shape: \n{content_df.shape[0]:,} rows × {content_df.shape[1]} columns\n")
print(f"Content CSV loaded | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# Specify Food & Compound CSV paths
# Change this to point to the CSV file
Food_CSV_PATH = r"C:\Users\Count_Dracula\FooDB\Food_FDB.csv"
Compound_CSV_PATH = r"C:\Users\Count_Dracula\FooDB\Compound_FDB_PubChemNames.csv"

# Quick existence check
assert os.path.exists(CSV_PATH), f"Food CSV file not found: {Food_CSV_PATH}"
print(f"Food CSV path is valid | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

assert os.path.exists(CSV_PATH), f"Compound CSV file not found: {Compound_CSV_PATH}"
print(f"Compound CSV path is valid | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# Reading Compound CSV file
print(f"Reading Compound CSV with pandas | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
print(f"It will take a while...\n")

# Check encoding
with open(Compound_CSV_PATH, "rb") as f:
    result = chardet.detect(f.read(100000))
print(f"Compound CSV file encoding: \n{result}\n")

# Read entire CSV
cpd_df = pd.read_csv(Compound_CSV_PATH, encoding=result["encoding"], low_memory=False)

print(f"Compound CSV shape: \n{cpd_df.shape[0]:,} rows × {cpd_df.shape[1]} columns\n")
print(f"Compound CSV loaded | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# Reading Food CSV file
print(f"Reading Food CSV with pandas | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"It will take a while...\n")

# Check encoding
with open(Food_CSV_PATH, "rb") as f:
    rawdata = f.read(100000)
    result = chardet.detect(rawdata)
print(f"Food CSV file encoding: \n{result}\n")

# Try detected encoding first
try:
    food_df = pd.read_csv(Food_CSV_PATH, encoding=result["encoding"], low_memory=False)
except UnicodeDecodeError:
    print(f"Detected encoding failed. Trying fallback encodings...\n")
    
    fallback_encodings = ["latin1", "cp1252", "ISO-8859-1"    ]

    for enc in fallback_encodings:
        try:
            print(f"Trying encoding: {enc}")

            food_df = pd.read_csv(Food_CSV_PATH, encoding=enc, low_memory=False)

            print(f"Successfully loaded using: {enc}\n")
            break
        except UnicodeDecodeError:
            print(f"Failed with: {enc}")

print(f"Food CSV shape: \n{food_df.shape[0]:,} rows × {food_df.shape[1]} columns\n")
print(f"Food CSV loaded | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# food_df cleanup
print(f"Renaming & dropping food_df columns | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# rename columns
food_df_rnmd = food_df.rename(columns={
    'public_id': 'public_food_id'
})

# drop unnecessary columns
food_df_drp = food_df_rnmd.drop(columns=[
    'wikipedia_id',
    'picture_file_name',
    'picture_content_type',
    'picture_file_size',
    'picture_updated_at',
    'created_at',
    'updated_at',
    'creator_id',
    'updater_id',
    'export_to_afcdb',
    'category',
    'export_to_foodb'
], errors='ignore')

print(f"Initial columns: \n{food_df.columns.tolist()}\n")
print(f"Remaining columns: \n{food_df_drp.columns.tolist()}\n")
print(f"Done renaming & dropping food_df columns | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# create harmonized food group (similar to CPDat product use category)
print(f"Creating & populating harmonized food group (similar to CPDat product use category) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# copy food_df_drp
food_df_harm = food_df_drp.copy()

HARMONIZED_SEPARATOR = " | "

# create column if not present
if "food_group_harmonized" not in food_df_harm.columns:
    food_df_harm["food_group_harmonized"] = None

# Move food_group_harmonized after food_type
cols = list(food_df_harm.columns)
if "food_type" in cols and "food_group_harmonized" in cols:
    cols.insert(
        cols.index("food_type") + 1,
        cols.pop(cols.index("food_group_harmonized"))
    )
    food_df_harm = food_df_harm[cols]

print(f"Created 'food_group_harmonized' col | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# helper: replace missing with literal "NaN"
def harmonized_value(x):
    return x if isinstance(x, str) and x.strip() else "NaN"

# populate puc_harmonized
food_df_harm["food_group_harmonized"] = food_df_harm.apply(
    lambda row: HARMONIZED_SEPARATOR.join([
        harmonized_value(row.get("food_group")),
        harmonized_value(row.get("food_subgroup")),
        harmonized_value(row.get("food_type")),
    ]),
    axis=1
)

print(f"Created & populated harmonized food group (similar to CPDat product use category) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# cpd_df cleanup
print(f"Renaming columns & replacing borken names with IUPAC names in cpd_df | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# drop unnecessary columns
cpd_df_drp = cpd_df.drop(columns=[
    'state',
    'annotation_quality',
    'kingdom'
], errors='ignore')

# detect mojibake / broken text in orig_feature_name & replace with iupac_name
def has_mojibake(text):
    if pd.isna(text):
        return False

    text = str(text)

    # common mojibake fragments
    mojibake_patterns = [
        r'Ã.',
        r'Â.',
        r'â..',
        r'�',
        r'¤',
        r'�'
    ]

    for pattern in mojibake_patterns:
        if re.search(pattern, text):
            return True

    # excessive non-ascii control-like artifacts
    weird_ratio = sum(ord(c) > 127 for c in text) / max(len(text), 1)

    if weird_ratio > 0.3:
        return True

    return False

cpd_df_moji = cpd_df_drp.copy()

# replace corrupted orig_feature_name with iupac_name
mask = cpd_df_drp['orig_feature_name'].apply(has_mojibake)

# print replacements before overwriting
print(f"{len(cpd_df_drp.loc[mask, 'orig_feature_name'])} broken compound names found \nReplacing with existing IUPAC names & IUPAC names from PubChem")

print("\nDetected mojibake replacements:\n")
for old_val, new_val in zip(
    cpd_df_drp.loc[mask, 'orig_feature_name'],
    cpd_df_drp.loc[mask, 'iupac_name']
):
    print(f"{old_val} >>> {new_val}")

# replace corrupted orig_feature_name with iupac_name
cpd_df_moji.loc[mask, 'orig_feature_name'] = (
    cpd_df_drp.loc[mask, ['iupac_name', 'iupac_name_pubchem', 'feature_name_pubchem']]
    .replace(r'^\s*$', pd.NA, regex=True)
    .bfill(axis=1)
    .iloc[:, 0]
)

print(f"\nInitial columns: \n{cpd_df.columns.tolist()}\n")
print(f"Remaining columns: \n{cpd_df_moji.columns.tolist()}\n")
print(f"Done renaming columns & replacing borken names with IUPAC names in cpd_df | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# cpd_df_moji cleanup
print(f"Removing rows with missing InChIKeys, renaming & resorting columns | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

cpd_df_clean = cpd_df_moji.copy()

print(f"Initial row count: {len(cpd_df_clean):,}\n")

# remove rows with missing inchikey
col_name = 'inchikey'

if col_name in cpd_df_clean.columns:
    rows_before = len(cpd_df_clean)

    # detect missing / empty values
    mask_remove = (cpd_df_clean[col_name].isna() |
        (cpd_df_clean[col_name]
            .astype(str)
            .str.strip()
            .isin(['', ' ', 'nan', 'None', 'NA', 'N/A'])))

    n_removed = mask_remove.sum()

    print(f"Rows with missing inchikey: {(n_removed):,}")

    cpd_df_clean = cpd_df_clean.loc[~mask_remove].copy()
    rows_after = len(cpd_df_clean)

    print(f"Rows before: {(rows_before):,}")
    print(f"Rows after: {(rows_after):,}")
else:
    print(f"Column '{col_name}' not found\n")


# rename PubChem columns
rename_dict = {
    'Name_PubChem': 'feature_name_pubchem',
    'IUPACName_PubChem': 'iupac_name_pubchem'
}

existing_rename_dict = {
    old: new
    for old, new in rename_dict.items()
    if old in cpd_df_clean.columns
}

if existing_rename_dict:
    print(f"\nRenaming columns:")
    print(existing_rename_dict)
    cpd_df_clean = cpd_df_clean.rename(columns=existing_rename_dict)
else:
    print(f"\nNo matching PubChem columns found for renaming\n")


# reorder PubChem columns
cols_to_place = ['orig_feature_name', 'feature_name_pubchem', 'iupac_name_pubchem']

if all(col in cpd_df_clean.columns for col in cols_to_place):
    cols = cpd_df_clean.columns.tolist()

    # remove columns to reposition
    cols.remove('feature_name_pubchem')
    cols.remove('iupac_name_pubchem')

    # insert feature_name_pubchem after orig_feature_name
    idx_orig = cols.index('orig_feature_name')
    cols.insert(idx_orig + 1, 'feature_name_pubchem')

    # insert iupac_name_pubchem after feature_name_pubchem
    idx_feat = cols.index('feature_name_pubchem')
    cols.insert(idx_feat + 1, 'iupac_name_pubchem')

    # reorder dataframe
    cpd_df_clean = cpd_df_clean[cols]

    print(f"PubChem columns reordered successfully\n")
else:
    print(f"Required columns for reordering not found\n")

print(f"Final row count: {len(cpd_df_clean):,}\n")
print(f"Final columns:")
print(cpd_df_clean.columns.tolist())

print(f"\nDone removing rows with missing InChIKeys, renaming & resorting columns | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# Remove chemicals by InChIKey, names, and keywords from cpd_df_clean
print(f"Removing features by InChIKey, names & keywords from cpd_df_clean | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

cpd_df_inchikey = cpd_df_clean.copy()

initial_n = len(cpd_df_inchikey)
print(f"Number of entries before InChIKey/name filtration: \n{initial_n:,}\n")


# List of InChIKeys to drop (e.g., mixtures, inorganic compounds & inorganic mixtures, etc.)
drop_inchikeys = [
    "ATJFFYVFTNAWJD-UHFFFAOYSA-N", "NINIDFKCEFEMDL-UHFFFAOYSA-N", "AKEJUJNQAAGONA-UHFFFAOYSA-N", # Tin; Sulfur; Sulfur trioxide
    "QAOWNCQODCNURD-UHFFFAOYSA-N", "QHMQWEPBXSHHLH-UHFFFAOYSA-N", "PXJJSXABGXMUSU-UHFFFAOYSA-N", # Sulfuric Acid; Sulfur tetrafluoride; Sulfur monochloride
    "SFZCNBIFKDRMGX-UHFFFAOYSA-N", "RAHZWNYVWXNFOC-UHFFFAOYSA-N", "LSNNMFCWUKXFEE-UHFFFAOYSA-M", # Sulfur Hexafluoride; Sulfur Dioxide; Hydrogen sulfite
    "QAOWNCQODCNURD-UHFFFAOYSA-L", "GEHJYWRUCIMESM-UHFFFAOYSA-L", "KEAYESYHFKHZAL-UHFFFAOYSA-N", # Sulfate; Sodium Sulfite; Sodium
    "ZLMJMSJWJFRBEC-UHFFFAOYSA-N", "OAICVXFJPJFONN-UHFFFAOYSA-N", "NBIIXXVUZAFLBC-UHFFFAOYSA-N", # Potassium; Phosphorus; Phosphoric acid
    "ABLZXFCXXLZCGV-UHFFFAOYSA-N", "UEZVMMHDMIWARA-UHFFFAOYSA-M", "NBIIXXVUZAFLBC-UHFFFAOYSA-K", # Phosphonic acid; Phosphonate; Phosphate
    "IJGRMHOSHXDMSA-UHFFFAOYSA-N", "GKAOGPIIYCISHV-UHFFFAOYSA-N", "PNDPGZBMCMUPRI-UHFFFAOYSA-N", # Nitrogen; Neon; Iodine
    "UFHFLCQGNIYNRP-UHFFFAOYSA-N", "PXGOKWXKJXAPGV-UHFFFAOYSA-N", "KRHYYFGTRYWZRS-UHFFFAOYSA-M", # Hydrogen; Fluorine; Fluoride
    "XFXPMWWXUTWYJX-UHFFFAOYSA-N", "IQPQWNKOIGAROB-UHFFFAOYSA-N", "IXBPPZBJIFNGJJ-UHFFFAOYSA-N", # Cyanide; Cyanate; Cyanamide,cyano-,sodium salt
    "XZMCDFZZKTWFGF-UHFFFAOYSA-N", "VEXZGXHMUGYJMC-UHFFFAOYSA-M", "BVKZGUZCCUSVTD-UHFFFAOYSA-L", # Cyanamide; Chloride; Carbonate
    "VNWKTOKETHGBQD-NJFSPNSNSA-N", "OKTJSMMVPCPJKN-UHFFFAOYSA-N", "GDTBXPJZTBHREO-UHFFFAOYSA-N", # Carbon-14; Carbon; Bromine
    "CPELXLSAUQHCOX-UHFFFAOYSA-M", "UXNBTDLSBQFMEH-UHFFFAOYSA-N", "QGZKDVFQNNGYKY-UHFFFAOYSA-O", # Bromide; Brass; Ammonium
    "QGZKDVFQNNGYKY-UHFFFAOYSA-N", "QTBSBXVTEAMEQO-UHFFFAOYSA-M", "DVARTQFDIMZBAA-UHFFFAOYSA-O", # Ammonia; Acetate; Ammonium nitrate
    "NLXLAEXVIDQMFP-UHFFFAOYSA-N", "BFNBIHQBYMNNAN-UHFFFAOYSA-N", "XLYOFNOQVPJJNP-UHFFFAOYSA-N", # Ammonium chloride; Ammonium sulfate; Water
    "QMMFVYPAHWMCMS-UHFFFAOYSA-N", "HEDRZPFGACZZDS-UHFFFAOYSA-N", "XYFCBTPGUUZFHI-UHFFFAOYSA-N", # Dimethyl sulfide; Chloroform; Phosphine
    "AQSJGOWTSHOLKH-UHFFFAOYSA-N", "USFZMSVCRYTOJT-UHFFFAOYSA-N", "AJPJDKMHJJGVTQ-UHFFFAOYSA-K", # Phosphite; Ammonium acetate; Sodium phosphate
    "SWLVFNYSXGMGBS-UHFFFAOYSA-N", "DDSPUNTXKUFWTM-UHFFFAOYSA-N", "UKFWSNCTAHXBQN-UHFFFAOYSA-N", # Ammonium bromide; Tin oxide; Ammonium iodide
    "HEMHJVSKTPXQMS-UHFFFAOYSA-M", "VHUUQVKOLVNVRT-UHFFFAOYSA-N", "AVXURJPOCDRRFD-UHFFFAOYSA-N", # Sodium hydroxide; Ammonium hydroxide; Hydroxylamine
    "LDDQLRUQCUTJBB-UHFFFAOYSA-N", "QXNVGIXVLWOKEQ-UHFFFAOYSA-N", "BTIJJDXEELBZFS-UHFFFAOYSA-K", # Ammonium fluoride; Disodium; Hemin
    "GYYDPBCUIJTIBM-DYOGSRDZSA-N", "XLYOFNOQVPJJNP-UHFFFAOYSA-M", "NHNBFGGVMKEFGY-UHFFFAOYSA-N", # Agar; Hydroxide; Nitrate
    "BVKZGUZCCUSVTD-UHFFFAOYSA-N", "JYYOBHFYCIDXHH-UHFFFAOYSA-N", "NJWBAJNKROKYMV-UHFFFAOYSA-N", # Carbonic acid; Carbonic acid--water (1/1); C 15
    "NBIIXXVUZAFLBC-ZRLBSURWSA-N", "CWYNVVGOOAEACU-UHFFFAOYSA-N", "UVDDHYAAWVNATK-VGKOASNMSA-L", # (3H)Phosphoric acid; Fe +2 ion; Tin, dibutylbis(2,4-pentanedionato-.kappa.O2,.kappa.O4)-, (OC-6-11)-
    "YVBGRQLITPHVOP-UHFFFAOYSA-L", "OPJWPPVYCOPDCM-UHFFFAOYSA-N", "UGTABSAUSFGUPH-UHFFFAOYSA-N", # Triphosphoric acid, sodium salt; Fatty acids, C16-18, 2-ethylhexyl esters; Fatty acids, C8-10, propylene esters
    "ZYBWTEQKHIADDQ-UHFFFAOYSA-N", "RSEKYLFCZNOVCI-UHFFFAOYSA-N", "KSBAEPSJVUENNK-UHFFFAOYSA-L", # Ethanol, mixt. with methanol; Lemon, ext.; Hexanoic acid, 2-ethyl-, tin(2+) salt
    "FXAZUIKRKXGTRK-UHFFFAOYSA-N", "YNPNZTXNASCQKK-LHNTUAQVSA-N", "YXFVVABEGXRONW-JGUCLWPXSA-N", # Disodium methanediylidenebis(azanide); Phenanthrene-d10; Toluene-d8
    "LRHPLDYGYMQRHN-NWURLDAXSA-N", "ISWSIDIOOBJBQZ-QNKSCLMFSA-N", "LQNUZADURLCDLV-RALIUCGRSA-N", # 1-Butanol-d10; Phenol-d6; Nitrobenzene-d5
    "OKKJLVBELUTLKV-MZCSYVLQSA-N", "WDECIBYCCFPHNR-AQZSQYOVSA-N", "UHOVQNZJYSORNB-MZWXYZOWSA-N", # Methan-d3-ol-d; Chrysene-d12; Benzene-d6
    "CWRYPZZKDGJXCA-WHUVPORUSA-N", "RSWGJHLUYNHPMX-UHFFFAOYSA-N", "NIXOWILDQLNWCW-UHFFFAOYSA-M", # Acenaphthene-d10; Tall-oil rosin; 2-Propenoic acid, ion(-1)
    "IWBUOAHJGJZERP-UHFFFAOYSA-N", "VBJGJHBYWREJQD-UHFFFAOYSA-M", "DJEHXEMURTVAOE-UHFFFAOYSA-M", # 2-Propenoic acid, butyl ester, polymer with diethenylbenzene and ethenylbenzene; Sodium phosphate, monobasic, dihydrate; Sulfurous acid, potassium salt (1:1)
    "PTFCDOFLOPIGGS-UHFFFAOYSA-N", "UNXRWKVEANCORM-UHFFFAOYSA-I", "OLGIDLDDXHSYFE-UHFFFAOYSA-N", # Zn2+; Triphosphate(5-); Tin (stannane)
    "DHCDFWKWKRSZHF-UHFFFAOYSA-L", "LSNNMFCWUKXFEE-UHFFFAOYSA-L", "UCKMPCXJQFINFW-UHFFFAOYSA-N", # Thiosulfate; Sulfite; Sulfide
    "MCAHWIHFGHIESP-UHFFFAOYSA-L", "NPYPAHLBTDXSSS-UHFFFAOYSA-N", "XHXFXVLFKHQFAL-UHFFFAOYSA-N", # Selenite; Potassium; Phosphoryl chloride
    "JRPHGDYSKGJTKZ-UHFFFAOYSA-N", "ABLZXFCXXLZCGV-UHFFFAOYSA-L", "CMFNMSMUKZHDEY-UHFFFAOYSA-M", # Phosphoroselenoic acid; Phosphite; Peroxynitrite
    "TWLXDPFBEPBAQB-UHFFFAOYSA-N", "QEHKBHWEUPXBCW-UHFFFAOYSA-N", "IOVCWXUNBOPUCH-UHFFFAOYSA-M", # Orthoperiodic acid; Nitrogen chloride (NCl3); Nitrite
    "XMBWDFGMSWQBCA-UHFFFAOYSA-M", "ICIWUVCWSCSTAQ-UHFFFAOYSA-M", "NBIIXXVUZAFLBC-UHFFFAOYSA-L", # Iodide; Iodate; Hydrogen phosphate
    "VEXZGXHMUGYJMC-UHFFFAOYSA-N", "OAKJQQAXSVQMHS-UHFFFAOYSA-N", "JPVYNHNXODAKFH-UHFFFAOYSA-N", # Hydrogen chloride; Hydrazine; Cu2+
    "VTLYFUHAOXGGBS-UHFFFAOYSA-N", "NBIIXXVUZAFLBC-UHFFFAOYSA-M", "OSVXSBDYLRYLIG-UHFFFAOYSA-N", # Fe3+; Dihydrogen phosphate; Chlorine oxide
    "XTEGARKTQYYJKE-UHFFFAOYSA-N", "UGFAIRIUMAVXCW-UHFFFAOYSA-N", "OWTFKEBRIAXSMO-UHFFFAOYSA-N", # Chlorate; Carbon monoxide; Arsenite
    "IIACRCGMVDHOTQ-UHFFFAOYSA-N", "AZDRQVAHHNSJOQ-UHFFFAOYSA-N", "HPGGPRDJHPYFRM-UHFFFAOYSA-J", # Amidosulfonic acid; Aluminum; Tin chloride (SnCl4)
    "XLJMAIOERFSOGZ-UHFFFAOYSA-M", "SERFXIDXLAJMRQ-UHFFFAOYSA-N", "AXZWODMDQAVCJE-UHFFFAOYSA-L", # Cyanate; Trimethylalkane (tris(methane) alumane); Tin(II) chloride (SnCl2)
    "KJSPVJJOPONRTK-UHFFFAOYSA-M", "FPAFDBFIGPHWGO-UHFFFAOYSA-N", "HWGNBUXHKFFFIH-UHFFFAOYSA-I", # Tetradecanoic acid Li salt; Talc; Sodium triphosphate (Na5(P3O10))
    "GGJFWMOVUFBSIN-YDBXVIPQSA-O", "UVSMMLABJBJNGH-WWJIDFMMSA-N", "LGMLJQFQKXPRGA-VPVMAENOSA-K", # Se-Adenosylselenomethionine; Se-Adenosylselenohomocysteine; Polyamines (gadolinium(3+) ion bis((2R,3R,4R,5S)-6-(methylamino)hexane-1,2,3,4,5-pentol) 2-[bis({2-[(carboxylatomethyl)(carboxymethyl)amino]ethyl})amino]acetate)
    "FKNQFGJONOIPTF-UHFFFAOYSA-N", "WAEMQWOKJMHJLA-UHFFFAOYSA-N", "JLVVSXFLKOJNIY-UHFFFAOYSA-N", # Na+; Mn2+; Mg2+
    "WQYVRQLZKVEZGA-UHFFFAOYSA-N", "ZCZCOXLLICTZAH-UHFFFAOYSA-N", "JUINSXZKUKVTMD-UHFFFAOYSA-N", # Hypochlorite; Hypothiocyanite; Hydrogen-azide
    "JZCCFEFSEZPSOG-UHFFFAOYSA-L", "BQPIGGFYSBELGY-UHFFFAOYSA-N", "GPRLSGONYQIRFK-UHFFFAOYSA-N", # Hydrocyanite; Hg2+; H+
    "RJGCOINDZWXYSL-UHFFFAOYSA-N", "KLMCZVJOEAUDNE-UHFFFAOYSA-N", "BDAGIHXWWSANSR-UHFFFAOYSA-M", # Gellan gum (tellanylium); FR (francium); formate
    "BWGNESOTFCXPMA-UHFFFAOYSA-N", "OXHQNTSSPHKCPB-UHFFFAOYSA-L", "KZBUYRJDOAKODT-UHFFFAOYSA-N", # Disulfides; D-Gluconic acid Mn(II) salt; Chlorine
    "PHMDYZQXPPOZDG-UHFFFAOYSA-N", "WLZRMCYVCSSEQC-UHFFFAOYSA-N" # Cer(d20:0/22:2(6Z,13Z)) (gallane); Cd2+
]

# Fool-proofing checks
required_cols = ['orig_feature_name', 'inchikey']

if all(col in cpd_df_inchikey.columns for col in required_cols):
    # Mask for rows to remove
    mask_drop = cpd_df_inchikey['inchikey'].isin(drop_inchikeys)

    dropped_df = cpd_df_inchikey.loc[mask_drop, ['orig_feature_name', 'inchikey']].drop_duplicates()

    print(f"Dropped compounds by InChIKey:")
    if len(dropped_df) > 0:
        for _, row in dropped_df.iterrows():
            print(f"{row['inchikey']} >>> {row['orig_feature_name']}")
    else:
        print(f"No matching InChIKeys found\n")

    # Drop rows
    cpd_df_inchikey = cpd_df_inchikey.loc[~mask_drop].copy()

    final_n = len(cpd_df_inchikey)

    print(f"\nRows removed by InChIKey: {initial_n - final_n:,}")
    print(f"Rows remaining: {final_n:,}\n")
else:
    print(f"Required columns not found:")
    print(required_cols)
    
# # Drop rows from CPDat where InChIKey matches any in the list
# cpd_df_inchikey = cpd_df_inchikey[~cpd_df_inchikey["inchikey"].isin(drop_inchikeys)]

# # Diagnostic print
# print(f"Dropped {len(drop_inchikeys):,} unique compounds by InChIKey. \n\nRemaining rows: \n{len(cpd_df_inchikey):,}\n")


# Define the unwanted chemicals regex (e.g., mixtures, inorganic compounds & inorganic mixtures, etc.)
unwanted_pattern = ("mica|talc|kerosine|kerosene|gasoline|aluminum|aluminium|alumina|barium|beryl|bismuth|"
                    "cadmium|calcium|cesium|caesium|cerium|ceric|chromium|cobalt|copper|cupric|cuprat|cupferron|"
                    "dysprosium|erbium|europium|gallium|gadolin|german|gold|hafnium|holmium|iron|indium|iridium|"
                    "lanthan|lead|lithium|lutetium|magnes|mangan|mercur|molybd|neodym|nickel|niobium|osmium|"
                    "pallad|praseodym|platin|rhodium|rhenium|rhenate|radon|radium|ruthenium|rubidium|neodym|"
                    "samarium|scandium|silver|strontium|titan|thorium|thallium|thallic|tantal|tellur|terbium|tungsten|tungstate|"
                    "thulium|tin fluoride|vanad|uranium|uranyl|yttrium|ytterbium|zinc|zircon|"                    
                    "diamond|ammonium tetraborate|ammonium hydrogen phosphate|glycosides|uprous oxide|cryolite|arsenate|"
                    "ferric|emery|ferrous ammonium sulfate|ferrosilicon|dibasic sodium phosphate|trona|ferrous ascorbate|"
                    "disodium phosphate dihydrate|disodium phosphate dodecahydrate|monosodium phosphate|sodium phosphate P 32|"
                    "trisodium phosphate|ammonium hydrogen sodium phosphate tetrahydrate|trisodium nitrilotriacetate|"
                    "foscarnet trisodium|fluorphlogopite|fluoboric acid|ferrous oxide|ferrous sulfide|ferrous chloride|"
                    "phosphate buffered saline|dichloromethane|diborane|boric acid|cristobalite|tritium|naphthalene d8|"
                    "deuterochloroform|deuterium|acid phosphate|aloe vera|air|activated charcoal|actinolite|"
                    "ammonium carbonate|ammonium carbamate|ammonium bisulfite|ammonium bisulfate|ferrlecit|"
                    "ammonium ferrous sulfate hexahydrate|dichromate|chromate|nitrogenase|"
                    "ammonium phosphate|ammonium perchlorate|ammonium isothiocyanate|hydroxytriphenylstannane|"
                    "ammonium iodate|ammonium hydrogen difluoride|ammonium hydrogen carbonate|"
                    "ammonium sulfite|ammonium sulfide|ammonium sulfamate|ammonium sodium hydrogenorthophosphate|"
                    "argon|antimony|antigorite|anhydrite|andalusite|anatase|amorphous silica|ammonium thiosulfate|"
                    "borax|borat|boran|spodumene|bauxite|bakelite|azanide|aventurine|attapulgite|arsine|arsenous|arsenic|"
                    "carbene|calcite|cacodylic acid|alkanes|alkenes|bromoform|iodoform|ftoroform|boron|boric|"
                    "carbon tetrafluoride|carbon tetrachloride|carbon monoxide|carbon disulfide|carbon dioxide|carbon black|"
                    "chlorine|charcoal|cellulase|cellulose|carbonyl sulfide|carbonic acid sodium salt|carbonic acid disodium salt|"
                    "cornmint oil|coal|chrysotile|chromite|chromic|chromate|chloromethane|chlorite|"
                    "alloy|plaster of paris|syenite|natural rubber|mullite|montmorillonite|mineral spirits|radical|"             
                    "potassium phosphate|potassium peroxide|aluminum phosphate|limnanthes alba|"
                    "lewisite|lazurite|langbeinite|kyanite|krypton|kaolinite|iodic acid|illite|yeast|"
                    "hydroxylamine sulfate|hydroxylamine hydrochloride|hydrotalcite|nitrogen oxides|"
                    "hydrogen sulfide|hydrogen selenide|hydrogen peroxide|hydrogen iodide|hydrogen fluoride|hydrogen cyanide|"
                    "hydrochloric acid|hydrobromic acid|huntite|hematite|arsinous|arsonate|"
                    "helium|hectorite|gypsum|graphite|grape seed oil|gram's iodine|goethite|fuming sulfuric acid|"
                    "periodic acid|perchloric acid|perboric acid|pentasodium triphosphate|ozone|oxygen|hydroxylapatite|"
                    "orange terpenes|olivine|nitrous oxide|nitrogen dioxide|nitric oxide|nitric acid|"
                    "perlite|zeolite|xenon|wollastonite|vitis vinifera|vermiculite|stainless steel|arsinate|"
                    "dolomite|sodium stannate|sodium phosphite|sodium octaborate|sodium peroxydicarbonate|"
                    "sodium monofluorophosphate|sodium methanediphosphonate|sulfur dioxide|borate|"
                    "sodium cyanodithioimidocarbonate|sodium arsenate|sodium methanearsonate|"
                    "tetrasodium pyrophosphate|terpenes|superoxide|starch|stannous|stannane|"
                    "sodium sulfide|sodium sulfate|sodium sulfanilate|sodium silicate|sodium pyrophosphate|"
                    "sodium potassium aluminum silicate|sodium polyphosphate|sodium polymetaphosphate|"
                    "sodium phosphinate|sodium persulfate|pertechnetate|perrhenate|"
                    "peroxoborate|sodium peroxide|sodium permanganate|sodium periodate|sodium perchlorate|"
                    "perborate|sodium oxide|vanadate|nitroprusside|sodium nitrite|sodium nitrate|fullerite|"
                    "monohydrogen phosphate heptahydrate|molybdate|sodium metaphosphate|metaborate|"
                    "sodium metabisulfite|sodium magnesium silicate|sodium iodide|sodium iodate|magnesate|"
                    "sodium hypochlorite|sodium hydrogen sulfate|sodium hydrogen difluoride|arsonite|"
                    "sodium hydride|sodium hexametaphosphate|sodium hexafluorosilicate|sodium formate|sodium fluoride|"
                    "sodium ferrocyanide|sodium dichromate|sodium chromate|sodium cyanide|sodium cobaltinitrite|"
                    "sodium chlorite|sodium chloride|sodium chlorate|sodium bisulfite|sodium bisulfate|"
                    "sodium bicarbonate|sodium azide|sodium arsenite|sodium aluminosilicate|sodium aluminate|"
                    "potassium tungstate|potassium tripolyphosphate|potassium trifluorido|potassium titanate|"
                    "potassium tin|potassium thiocyanate|potassium tetrafluoroborate|potassium superoxide|"
                    "potassium sulfite|potassium sulfate|potassium pyrosulfate|potassium pyrophosphate|"
                    "potassium polysulfide|potassium phosphate|potassium persulfate|potassium peroxymonosulfate|"
                    "potassium permanganate|potassium periodate|potassium perchlorate|potassium perfluorooctanesulfonate|"
                    "potassium pentaborate tetrahydrate|potassium oxide|potassium nitrite|potassium nitrate|"
                    "potassium metaphosphate|potassium metaborate|potassium metabisulfite|potassium iodide|"
                    "potassium iodate|potassium hydroxide|potassium hydrogen monopersulfate|potassium hydrogen diiodate|"
                    "potassium hydride|potassium hexafluorophosphate|potassium hexacyanoferrate|potassium fluoride|"
                    "potassium ferricyanide|potassium dichromate|potassium bichromate|potassium cyanide|"
                    "potassium chromate|potassium chloride|potassium chlorate|potassium carbonate|potassium bromide|"
                    "potassium bromate|potassium borate|potassium bisulfate|potassium bicarbonate|"
                    "potassium arsenate|phosphorus oxide|phosphorus trifluoride|phosphotungstic acid|phosphorus tribromide|"
                    "phosphorus pentachloride|phosphorus pentasulfide|phosphorous trichloride|phosphorous acid|"
                    "phosphoric trichloride|phosphomolybdic acid|metaphosphoric acid|"
                    "silica|silyl|siloxane|silicon|silicic acid|silicate|silan|silicic|selenium|selenious|selenic|"
                    "extract|sapphire|sand|rose oil|aluminosilicate|diopside|quartz|pyrophyllite|borane|"
                    "almandite and pyrope garnet|aluplex|americium 241|ammonium cyanide|amosite|amylase|azurite|"
                    "boehmite|bovine insulin|cerous chloride heptahydrate|chlorous acid|clinoptilolite|"
                    "collagens|cyclic amp|epoxy resin|kinase|lactoferrin|lignin|limonite|"
                    "ferrate|ferrocene|ferrocyanide|ferroin|ferrous gluconate|ferrous lactate|ferrous oxalate|"
                    "ginseng|human chorionic gonadotropin|human insulin|hydrocarbon oils|invert sugar|"
                    "iodine chloride|iodine pentafluoride|iodine pentoxide|iodohippurate sodium|"
                    "hexasodium metaphosphate (p6o186(-))|potassium triphosphate (k5(p3o10))|"
                    "15n acid solution|nitrogen mustard hydrochloride|nitrogen tetroxide|nitrogen trifluoride|"
                    "pectin|peroxidase|phenoxy resin|phosphorodithioic acid|polyamide fibers|53249257|activated carbon|"
                    "pyrite|riebeckite|saponins|sepiolite|smithsonite|sodium bromate|sodium bromide|sodium selenate|"
                    "sodium selenite|sodium sulfamate|sodium thiocyanate|sodium thiosulfate|trimethylselenonium|seleno|"
                    "sodium trimetaphosphate|sterols|sulfurous acid|sulfuryl chloride|sulfuryl fluoride|terpene resin|"
                    "tetraphenyl tin|thulium|thulium oxide|tungstic acid|uraninite|phytosterols|selenide|selenate|"
                    "tocopherols|topaz|tremolite|tridymite|triethyltin bromide|triphosphoric acid|sodium aluminium phosphate|"
                    "sodium sulfate decahydrate|sodium sulfate|sodium silicate (na2sio3)|sodium silicate|sodium sesquicarbonate|"
                    "sodium phosphinate|sodium phosphate (na3po4)|sodium phosphate (na(h2po4))|sodium nitrite|sodium nitrate (nano3)|"
                    "sodium methyl sulfate|sodium metaphosphate|sodium magnesium aluminosilicate|sodium iron versenate|sodium hypochlorite|"
                    "sodium hydroxide (naoh)|sodium hydrogen sulfite|sodium hydrogen sulfate|sodium hydrogen carbonate|sodium formate|"
                    "sodium fluoride (naf)|sodium ferulate|sodium ferricitropyrophosphate|sodium dithionite (na2s2o4)|sodium disulfite|"
                    "sodium diphosphate (na4(p2o7))|sodium copper chlorophyllin|sodium chlorite|sodium chloride (nacl)|sodium carbonate|"
                    "sodium calcium edetate|sodium calcium aluminosilicate, hydrated|sodium borohydride|sodium aluminum silicate|"
                    "sodium aluminium oxide (naalo2)|sodium alum|potassium triphosphate (k5(p3o10))|potassium tetraborate|potassium sulfite (k2so3)|"
                    "potassium sulfate (k2so4)|potassium sorbate|potassium salts of fatty acids|potassium propionate|potassium phosphinate|"
                    "potassium phosphate (k3po4)|potassium phosphate (k(h2po4))|potassium persulfate (k2s2o8)|potassium permanganate|potassium palmitate|"
                    "potassium oxide|potassium nitrite (kno2)|potassium nitrate (kno3)|potassium metaphosphate|potassium iodide|"
                    "potassium iodate(v) (kio3)|potassium hypophosphate|potassium hydroxide (koh)|potassium hydrogen sulfate (khso4)|potassium hydrogen carbonate|"
                    "potassium disulfite|potassium diphosphate (k4(p2o7))|potassium chloride|potassium carbonate (k2co3)|potassium bromide (kbr)|"
                    "potassium bromate(v) (kbro3)|potassium bisulfite|potassium acid pyrophosphate|polymeric potassium metaphosphate|"
                    "disodium tetraborate|disodium sulfide|disodium pyrophosphate|disodium phosphate|dicalcium orthosilicate|potassium ferrocyanide|"
                    "diammonium sulfite|diammonium sulfide|diammonium hydrogen phosphate ((nh4)2hpo4)|ammonium sulfate|ammonium peroxydisulfate|"
                    "ammonium hydroxide|ammonium dihydrogenphosphate|ammonium chloride ((nh4)cl)|ammonium carbonate|ammonium bicarbonate|ammonium alum|"                    
                   )

exception_terms = [
    "Sandoricin",
    "Sandosapogenol",
    "Sandosaponin A",
    "Sandosaponin B",
    "Silandrin",
    "ent-Sandaracopimaradiene",
    "alpha-Irone"    
]

# Original unwanted pattern string
raw_pattern = unwanted_pattern

# Split into individual terms
terms = raw_pattern.split("|")

# Apply safer regex construction:
# - short/generic tokens → word bounded
# - multi-word phrases → keep as is
safe_terms = []
for t in terms:
    t = t.strip()
    if not t:
        continue
    # heuristic: single word without space → bound it
    if " " not in t:
        safe_terms.append(rf"\b{re.escape(t)}") # escape special regex characters: (, ), [, ], +, ., ?
    else:
        safe_terms.append(re.escape(t)) # escape special regex characters: (, ), [, ], +, ., ?

# Rebuild regex
safe_pattern = "|".join(safe_terms)

# Get all unique chemical names before filtering
all_unique_chems = cpd_df_inchikey["orig_feature_name"].dropna().unique()
print(f"Number of all unique compound names: \n{len(all_unique_chems):,}\n")

# Identify unique chemical names to drop
dropped_chems = []

for chem in all_unique_chems:
    chem_str = str(chem)

    # skip protected compounds
    if any(
        exc.lower() in chem_str.lower()
        for exc in exception_terms
    ):
        continue

    # normal unwanted matching
    if re.search(
        safe_pattern,
        chem_str,
        flags=re.IGNORECASE
    ):
        dropped_chems.append(chem)

# Print unique chemical names to drop
print(f"Number of dropped unique compound names: \n{len(dropped_chems):,}")
print(f"\nDropped unique compound names:")
for chem in sorted(dropped_chems):
    print(chem)

# Drop mixtures, inorganic compounds & inorganic mixtures, etc. by name/keyword
cpd_df_fltrd = cpd_df_inchikey.copy()
mask_unwanted = (
    cpd_df_inchikey["orig_feature_name"]
    .str.contains(
        safe_pattern,
        case=False,
        na=False,
        regex=True
    )
)

col = cpd_df_inchikey["orig_feature_name"].fillna("").astype(str).str.lower()
mask_exception = False

for exc in exception_terms:
    mask_exception = mask_exception | col.str.contains(exc.lower(), na=False)

# Remove unwanted unless protected by exception
cpd_df_fltrd = cpd_df_inchikey[~(mask_unwanted & ~mask_exception)].copy()


print(f"\nFiltered cpd_df_clean: \n{initial_n:,} → {len(cpd_df_fltrd):,} rows\n")
print(f"Done removing features by InChIKey, names & keywords from cpd_df_clean | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# content_df cleanup
print(f"Renaming & dropping content_df columns | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# rename columns
content_df_rnmd = content_df.rename(columns={
    'id': 'cont_id',
    'source_id': 'cpd_id',
    'orig_source_name': 'orig_feature_name'
})

print(f"content_df columns renamed\n")

# drop unnecessary columns
content_df_drp = content_df_rnmd.drop(columns=[
    'orig_citation',
    'citation',
    'citation_type',
    'creator_id',
    'updater_id',
    'created_at',
    'updated_at',
    'orig_method',
    'orig_unit_expression',
    'standard_content',
    'preparation_type',
    'export'
], errors='ignore')

print(f"content_df unnecessary columns dropped\n")
print(f"Initial columns: \n{content_df.columns.tolist()}\n")
print(f"Remaining columns: \n{content_df_drp.columns.tolist()}\n")
print(f"Done renaming & dropping content_df columns | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# content_df_drp cleanup
print(f"Dropping rows where source_type == 'Nutrient' & orig_feature_name contains 'total' | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

content_df_nutr = content_df_drp.copy()

print(f"Initial row count: {len(content_df_nutr):,}\n")

# remove rows where source_type == 'Nutrient'
col_name = 'source_type'
target_value = 'Nutrient'

if col_name in content_df_nutr.columns:
    rows_before = len(content_df_nutr)

    # ensure string-safe comparison
    mask_remove = (
        content_df_nutr[col_name]
        .astype(str)
        .str.strip()
        .eq(target_value)
    )

    n_removed = mask_remove.sum()
    content_df_nutr = content_df_nutr.loc[~mask_remove].copy()
    content_df_nutr = content_df_nutr.drop(columns=['source_type'], errors='ignore')
    rows_after = len(content_df_nutr)

    print(f"Removed rows where {col_name} == '{target_value}': {(n_removed):,}")
    print(f"Rows before: {(rows_before):,}")
    print(f"Rows after: {(rows_after):,}\n")
else:
    print(f"\nColumn '{col_name}' not found. Skipping cleanup.\n")


# remove rows where orig_feature_name contains 'total'
content_df_ttl = content_df_nutr.copy()

col_name_2 = 'orig_feature_name'
target_patterns = [
    'Carbohydrates, total',
    'Carbohydrates, total available',
    'Fat, total (Lipids)',
    'Fatty acids, total monounsaturated',
    'Fatty acids, total mono-unsaturated',
    'Fatty acids, total polyunsaturated',
    'Fatty acids, total poly-unsaturated',
    'Fatty acids, total saturated',
    'Sugars, total',
    'Fiber, total dietary',
    'Fiber, total dietary (AOAC)',
    'Protein, total',
    'Protein, total-N',
    'Total lipid (fat)',
]

if col_name_2 in content_df_ttl.columns:
    rows_before_2 = len(content_df_ttl)

    # Escape all patterns safely (removes regex meaning of parentheses, commas, etc.)
    safe_patterns = [re.escape(p) for p in target_patterns]

    # Build regex pattern from list
    regex_pattern = '|'.join(re.escape(p) for p in target_patterns)

    # Case-insensitive contains check
    # Vectorized match
    mask_remove = (
        content_df_ttl[col_name_2]
        .fillna("")
        .astype(str)
        .str.contains(regex_pattern, case=False, na=False, regex=True)
    )

    n_removed_2 = mask_remove.sum()

    # print matched examples
    print(f"Examples of removed rows:")
    print(content_df_ttl.loc[mask_remove, col_name_2].drop_duplicates())

    content_df_ttl = content_df_ttl.loc[~mask_remove].copy()
    rows_after_2 = len(content_df_ttl)

    print(f"\nRemoved rows: {(n_removed_2):,} \ncontaining: \n'{target_patterns}'\n")   
    print(f"Rows before: {(rows_before_2):,}")
    print(f"Rows after: {(rows_after_2):,}\n")
else:
    print(f"\nColumn '{col_name_2}' not found. Skipping cleanup.\n")

print(f"Final row count: {len(content_df_ttl):,}\n")
print(f"Rows where source_type == 'Nutrient' & orig_feature_name contains 'total' dropped | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# Populating content_df_ttl & cleanup
print(f"Populating food_name & orig_feature_name columns missing values | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

content_df_ppltd = content_df_ttl.copy()

print(f"Initial rows in content_df_ppltd: {len(content_df_ppltd):,}\n")
print(f"Initial columns: \n{content_df_ppltd.columns.tolist()}\n")

# fill orig_food_common_name using within-group values, then fallback to food_df_harm
col = 'orig_food_common_name'
key = 'food_id'

# baseline empty count (before any fill)
if col in content_df_ppltd.columns:
    empty_before = (
        content_df_ppltd[col].isna() |
        (content_df_ppltd[col].astype(str).str.strip() == '')
    ).sum()
    print(f"Empty '{col}' before filling: {empty_before:,}\n")
else:
    print(f"Missing column: {col}\n")

# within food_id fill (use any existing non-empty value)
if col in content_df_ppltd.columns and key in content_df_ppltd.columns:
    rows_before = len(content_df_ppltd)

    # normalize empties
    content_df_ppltd[col] = content_df_ppltd[col].replace(r'^\s*$', pd.NA, regex=True)

    # propagate first valid value within each food_id group
    content_df_ppltd[col] = content_df_ppltd.groupby(key)[col].transform(lambda s: s.ffill().bfill())

    print(f"Within-group fill completed")
else:
    print(f"Missing columns: {col} / {key}\n")

# fallback from food_df_harm.food_name
fallback_col = 'food_name'

if (col in content_df_ppltd.columns and
    key in content_df_ppltd.columns and
    fallback_col in food_df_harm.columns):
    rows_before = len(content_df_ppltd)

    # build lookup from food_df_harm
    food_lookup = (food_df_harm[[key, fallback_col]]
        .drop_duplicates(subset=key)
        .set_index(key)[fallback_col])

    # identify still-missing rows
    mask_missing = content_df_ppltd[col].isna() | (content_df_ppltd[col].astype(str).str.strip() == '')
    content_df_ppltd.loc[mask_missing, col] = content_df_ppltd.loc[mask_missing, key].map(food_lookup)
    n_filled = (~content_df_ppltd[col].isna()).sum()

    print(f"Fallback fill from food_df_harm completed")
else:
    print(f"Missing columns for fallback step")

# final empty count + filled summary
if col in content_df_ppltd.columns:
    empty_after = (content_df_ppltd[col].isna() |
        (content_df_ppltd[col].astype(str).str.strip() == '')).sum()

    filled_total = empty_before - empty_after

    print(f"Empty '{col}' after filling: {empty_after:,}")
    print(f"Total filled values: {filled_total:,}\n")
    

# populate/create food_name from food_df_harm using food_id
required_cols_content = ['food_id']
required_cols_food = ['food_id', 'food_name']

if (all(col in content_df_ppltd.columns for col in required_cols_content)
    and
    all(col in food_df_harm.columns for col in required_cols_food)):
    # Create lookup dictionary
    food_lookup = (food_df_harm[['food_id', 'food_name']]
        .drop_duplicates(subset='food_id')
        .set_index('food_id')['food_name'])

    # Populate/create food_name column
    content_df_ppltd['food_name'] = (content_df_ppltd['food_id'].map(food_lookup))
    n_food_names = content_df_ppltd['food_name'].notna().sum()

    print(f"food_name column created & populated")
    print(f"Matched food names: {n_food_names:,}\n")
else:
    print(f"Required columns missing for food_name population\n")


# populate empty orig_feature_name from cpd_df_fltrd using cpd_id
required_cols_content = ['cpd_id', 'orig_feature_name']
required_cols_cpd = ['cpd_id', 'orig_feature_name']

if (all(col in content_df_ppltd.columns for col in required_cols_content)
    and
    all(col in cpd_df_fltrd.columns for col in required_cols_cpd)):

    rows_before = len(content_df_ppltd)

    # Create lookup dictionary
    cpd_lookup = (cpd_df_fltrd[['cpd_id', 'orig_feature_name']]
        .drop_duplicates(subset='cpd_id')
        .set_index('cpd_id')['orig_feature_name'])

    # Detect empty/missing orig_feature_name values
    mask_empty = (content_df_ppltd['orig_feature_name'].isna() |
        (content_df_ppltd['orig_feature_name']
            .astype(str)
            .str.strip()
            .isin(['', 'nan', 'None', 'NA', 'N/A'])))

    n_empty_before = mask_empty.sum()

    # Populate only empty rows
    content_df_ppltd.loc[mask_empty, 'orig_feature_name'] = (
        content_df_ppltd.loc[mask_empty, 'cpd_id']
        .map(cpd_lookup))

    # Recount remaining empty values
    mask_empty_after = (content_df_ppltd['orig_feature_name'].isna() |
        (content_df_ppltd['orig_feature_name']
            .astype(str)
            .str.strip()
            .isin(['', 'nan', 'None', 'NA', 'N/A'])))

    n_empty_after = mask_empty_after.sum()

    print(f"orig_feature_name population complete")
    print(f"Empty values before: {n_empty_before:,}")
    print(f"Empty values after: {n_empty_after:,}")
    print(f"Filled values: {n_empty_before - n_empty_after:,}\n")
else:
    print(f"Required columns missing for orig_feature_name population")
    

# Move food_name after food_id & cpd_id before orig_feature_name
print(f"Initial columns before moving: \n{content_df_ppltd.columns.tolist()}\n")

required_cols = ['food_id', 'food_name', 'cpd_id', 'orig_feature_name']

if all(col in content_df_ppltd.columns for col in required_cols):
    cols = content_df_ppltd.columns.tolist()

    # Remove food_name from current position
    cols.remove('food_name')
    # Insert after food_id
    idx_food_id = cols.index('food_id')
    cols.insert(idx_food_id + 1, 'food_name')
    
    # Remove cpd_id from current position
    cols.remove('cpd_id') 
    # Insert before orig_feature_name
    idx_orig_feature_name = cols.index('orig_feature_name')
    cols.insert(idx_orig_feature_name, 'cpd_id')

    # Reorder dataframe
    content_df_ppltd = content_df_ppltd[cols]

    print(f"food_name & cpd_id moved\n")
else:
    print(f"Required columns not found for reordering\n")

print(f"Final columns: \n{content_df_ppltd.columns.tolist()}\n")
print(f"Final row count: {len(content_df_ppltd):,}\n")

print(f"Done populating food_name & orig_feature_name columns missing values | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# Further filtering of content_df_ppltd
print(f"Dropping rows with missing values in food_name, orig_feature_name, orig_content & cpd_id | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

content_df_fltrd = content_df_ppltd.copy()

print(f"Initial row count: {len(content_df_fltrd):,}\n")

# drop rows with empty/missing food_name
col_name = 'food_name'

if col_name in content_df_fltrd.columns:
    rows_before = len(content_df_fltrd)

    mask_remove = (content_df_fltrd[col_name].isna() |
        (content_df_fltrd[col_name]
            .astype(str)
            .str.strip()
            .isin(['', ' ', 'nan', 'None', 'NA', 'N/A'])))

    n_removed = mask_remove.sum()
    content_df_fltrd = content_df_fltrd.loc[~mask_remove].copy()
    rows_after = len(content_df_fltrd)

    print(f"Removed empty '{col_name}' rows: {n_removed:,}")
    print(f"Rows before: {rows_before:,}")
    print(f"Rows after: {rows_after:,}\n")
else:
    print(f"Column '{col_name}' not found\n")


# drop rows with empty/missing orig_feature_name
col_name = 'orig_feature_name'

if col_name in content_df_fltrd.columns:

    rows_before = len(content_df_fltrd)

    mask_remove = (content_df_fltrd[col_name].isna() |
        (content_df_fltrd[col_name]
            .astype(str)
            .str.strip()
            .isin(['', ' ', 'nan', 'None', 'NA', 'N/A'])))

    n_removed = mask_remove.sum()
    content_df_fltrd = content_df_fltrd.loc[~mask_remove].copy()
    rows_after = len(content_df_fltrd)

    print(f"Removed empty '{col_name}' rows: {n_removed:,}")
    print(f"Rows before: {rows_before:,}")
    print(f"Rows after: {rows_after:,}\n")
else:
    print(f"Column '{col_name}' not found\n")


# drop rows with no matching cpd_id in cpd_df_fltrd
required_cols_content = ['cpd_id']
required_cols_cpd = ['cpd_id']

if (all(col in content_df_fltrd.columns for col in required_cols_content)
    and
    all(col in cpd_df_fltrd.columns for col in required_cols_cpd)):
    rows_before = len(content_df_fltrd)

    # Valid cpd_ids from cpd_df_fltrd
    valid_cpd_ids = set(cpd_df_fltrd['cpd_id'])

    # Keep only matching rows
    mask_keep = content_df_fltrd['cpd_id'].isin(valid_cpd_ids)
    n_removed = (~mask_keep).sum()
    content_df_fltrd = content_df_fltrd.loc[mask_keep].copy()
    rows_after = len(content_df_fltrd)

    print(f"Removed rows with unmatched cpd_id: {n_removed:,}")
    print(f"Rows before: {rows_before:,}")
    print(f"Rows after: {rows_after:,}\n")
else:
    print(f"Required cpd_id columns not found\n")


# drop rows with empty/missing/zero orig_content
col_name = 'orig_content'

if col_name in content_df_fltrd.columns:
    rows_before = len(content_df_fltrd)

    # Empty/missing values
    mask_empty = (content_df_fltrd[col_name].isna() |
        (content_df_fltrd[col_name]
            .astype(str)
            .str.strip()
            .isin(['', ' ', 'nan', 'None', 'NA', 'N/A'])))

    # Numeric zero values
    mask_zero = (pd.to_numeric(content_df_fltrd[col_name], errors='coerce') == 0)

    # Print counts before removal
    n_empty = mask_empty.sum()
    n_zero = mask_zero.sum()

    print(f"Empty/missing '{col_name}' rows: {n_empty:,}")
    print(f"Zero-value '{col_name}' rows: {n_zero:,}\n")

    # Combined removal mask
    mask_remove = mask_empty | mask_zero
    n_removed = mask_remove.sum()
    content_df_fltrd = content_df_fltrd.loc[~mask_remove].copy()
    rows_after = len(content_df_fltrd)

    print(f"Removed empty/zero '{col_name}' rows: {n_removed:,}")
    print(f"Rows before: {rows_before:,}")
    print(f"Rows after: {rows_after:,}\n")
else:
    print(f"Column '{col_name}' not found\n")
    

print(f"Final row count: {len(content_df_fltrd):,}\n")
print(f"Done dropping rows with missing values in food_name, orig_feature_name, orig_content & cpd_id | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# Show stats
print(f"Showing stats | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

def df_stats(df, df_name):
    stats = pd.DataFrame({
        'column': df.columns,
        'dtype': [df[c].dtype for c in df.columns],
        'n_unique': [df[c].nunique(dropna=True) for c in df.columns],
        'n_missing': [df[c].isna().sum() for c in df.columns],
        'pct_missing': [
            round(df[c].isna().mean() * 100, 2)
            for c in df.columns
        ]
    })

    print(f"\n{'='*80}")
    print(f"{df_name}")
    print(f"{'='*80}")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns):,}")

    display(stats.sort_values(
        by='n_unique',
        ascending=False
    ))

# Run stats
df_stats(food_df_harm, "food_df_harm")
df_stats(cpd_df_fltrd, "cpd_df_fltrd")
df_stats(content_df_fltrd, "content_df_fltrd")

print(f"Done showing stats | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# Save filtered DFs
print(f"Writing filtered DFs to CSV | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# Save merged FooDB
food_df_harm.to_csv("Food_Fltrd.csv", index=False)
print(f"Done writing filtered Food DF to CSV | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

cpd_df_fltrd.to_csv("Compound_Fltrd.csv", index=False)
print(f"Done writing filtered Compound DF to CSV | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

content_df_fltrd.to_csv("Content_Fltrd.csv", index=False)
print(f"Done writing filtered Content DF to CSV | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print(f"All filtered DFs written to CSV | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# Merge all dfs
print(f"Merging filtered DFs into foodb_df | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# rename overlapping columns before merging
print(f"Renaming overlapping columns before merging | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

food_df_merge = food_df_harm.rename(columns={
    'food_name': 'food_name_food'
})

cpd_df_merge = cpd_df_fltrd.rename(columns={
    'orig_feature_name': 'orig_feature_name_cpd'
})

content_df_merge = content_df_fltrd.rename(columns={
    'food_name': 'food_name_cont',
    'orig_feature_name': 'orig_feature_name_cont'
})


# merge content_df with food_df using food_id
foodb_df_cont_food = content_df_merge.merge(
    food_df_merge,
    on='food_id',
    how='left'
)

print(f"Columns content_df + food_df: \n{foodb_df_cont_food.columns.tolist()}\n")

# merge resulting df with cpd_df using cpd_id
foodb_df = foodb_df_cont_food.merge(
    cpd_df_merge,
    on='cpd_id',
    how='left'
)

print(f"Columns after merge of all DBs: \n{foodb_df.columns.tolist()}\n")


# reorder columns (if columns exist)
print(f"Reordering columns | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

col_pairs = [
    ('food_id', 'food_name_food'),
    ('cpd_id', 'public_cpd_id'),
    ('inchikey', 'orig_feature_name_cpd'),
    ('iupac_name', 'iupac_name_pubchem'),
    ('orig_food_common_name', 'food_name_scientific')
]

cols = foodb_df.columns.tolist()

for first_col, second_col in col_pairs:
    if first_col in cols and second_col in cols:
        # remove the first column from current position
        cols.remove(first_col)
        # recompute index of second column
        idx_second = cols.index(second_col)
        # insert first column immediately before second column
        cols.insert(idx_second, first_col)

        print(f"Moved '{first_col}' before '{second_col}'")
    else:
        print(f"Skipping: '{first_col}' or '{second_col}' not found\n")

# Apply new column order
foodb_df = foodb_df[cols]

print(f"\nColumn reordering complete | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
print(f"Columns after reordering: \n{foodb_df.columns.tolist()}\n")


# create and populate food_sub_id
required_cols = ['food_id', 'orig_food_common_name']

if all(col in foodb_df.columns for col in required_cols):
    rows_before = len(foodb_df)

    # normalize names for grouping
    foodb_df['orig_food_common_name'] = (
        foodb_df['orig_food_common_name']
        .fillna('')
        .astype(str)
        .str.strip()
    )

    # build mapping within each food_id separately
    food_subid_map = {}

    for food_id, grp in foodb_df.groupby('food_id', sort=False):
        unique_names = (
            pd.Series(grp['orig_food_common_name'].unique())
            .loc[lambda x: x != '']
            .tolist()
        )

        # number from 1 within this food_id only
        for i, name in enumerate(unique_names, start=1):
            food_subid_map[(food_id, name)] = f"{food_id}_{i:02d}"

    # populate food_sub_id
    foodb_df['food_sub_id'] = [
        food_subid_map.get(
            (food_id, food_name),
            f"{food_id}_01"
        )
        for food_id, food_name in zip(
            foodb_df['food_id'],
            foodb_df['orig_food_common_name']
        )
    ]

    # Move food_sub_id after food_id
    cols = foodb_df.columns.tolist()

    if 'food_sub_id' in cols:
        cols.remove('food_sub_id')
        idx_food_id = cols.index('food_id')
        cols.insert(idx_food_id + 1, 'food_sub_id')
        foodb_df = foodb_df[cols]

    print(f"food_sub_id created and populated | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
else:
    print(f"Missing required columns: {required_cols}\n")

print(f"Final columns: \n{foodb_df.columns.tolist()}\n")
print(f"foodb_df shape: \n{foodb_df.shape[0]:,} rows × {foodb_df.shape[1]} columns\n")
print(f"Merged 3 filtered DFs into foodb_df | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# Show stats - Merged FooDB
print(f"Showing merged FooDB stats | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

def df_stats(df, df_name):
    stats = pd.DataFrame({
        'column': df.columns,
        'dtype': [df[c].dtype for c in df.columns],
        'n_unique': [df[c].nunique(dropna=True) for c in df.columns],
        'n_missing': [df[c].isna().sum() for c in df.columns],
        'pct_missing': [
            round(df[c].isna().mean() * 100, 2)
            for c in df.columns
        ]
    })

    print(f"\n{'='*80}")
    print(f"{df_name}")
    print(f"{'='*80}")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns):,}")

    display(stats)

# Run stats
df_stats(foodb_df, "Merged foodb_df")

print(f"Done showing merged FooDB stats | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# Save merged FooDB
print(f"Writing merged FooDB to CSV/Excel | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"It will take a while...\n")

# # Save merged FooDB as Excel
# foodb_df.to_excel("FooDB_Merged_XL.xlsx", index=False, engine="openpyxl")
# print(f"Merged FooDB written to Excel | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# Save merged FooDB as CSV (way faster)
foodb_df.to_csv("FooDB_Merged.csv", index=False, sep="\t")
print(f"Merged FooDB written to CSV | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

